# Exact inference using a Junction Tree

This example demonstrates how to construct a Junction Tree, define clique potentials and perform exact inference using belief propagation.

In [23]:
import numpy as np
from pgmpy.models import JunctionTree
from pgmpy.factors.discrete import DiscreteFactor
from pgmpy.inference import BeliefPropagation

In [8]:
# Initializing the Junction Tree
jt = JunctionTree()
print(jt)

JunctionTree with 0 nodes and 0 edges


In [9]:
# Defining Cliques
clique_ab = ("A", "B")
clique_bc = ("B", "C")
clique_cd = ("C", "D")

# Adding Nodes
jt.add_nodes_from([clique_ab, clique_bc, clique_cd])

In [10]:
print(f"Nodes: ",jt.nodes())

Nodes:  [('A', 'B'), ('B', 'C'), ('C', 'D')]


In [11]:
# Adding Edges
jt.add_edges_from([
    (clique_ab, clique_bc),
    (clique_bc, clique_cd)
])

In [12]:
print(f"Edges: ",jt.edges())

Edges:  [(('A', 'B'), ('B', 'C')), (('B', 'C'), ('C', 'D'))]


In [13]:
phi_ab = DiscreteFactor(
    variables=["A", "B"],
    cardinality=[2, 2],
    values=[
        0.3, 0.7,
        0.8, 0.2
    ]
)

phi_bc = DiscreteFactor(
    variables=["B", "C"],
    cardinality=[2, 2],
    values=[
        0.6, 0.4,
        0.5, 0.5
    ]
)

phi_cd = DiscreteFactor(
    variables=["C", "D"],
    cardinality=[2, 2],
    values=[
        0.9, 0.1,
        0.2, 0.8
    ]
)

In [14]:
print(phi_ab)

+------+------+------------+
| A    | B    |   phi(A,B) |
+======+======+============+
| A(0) | B(0) |     0.3000 |
+------+------+------------+
| A(0) | B(1) |     0.7000 |
+------+------+------------+
| A(1) | B(0) |     0.8000 |
+------+------+------------+
| A(1) | B(1) |     0.2000 |
+------+------+------------+


In [15]:
jt.add_factors(phi_ab, phi_bc, phi_cd)

In [16]:
print(f"Model Validity: {jt.check_model()}")

Model Validity: True


In [18]:
bp = BeliefPropagation(jt)

In [19]:
# Query for the marginal probability of C
marginal_c = bp.query(variables=["C"])
print(f"Marginal P(C): ",marginal_c)

Marginal P(C):  +------+----------+
| C    |   phi(C) |
+======+==========+
| C(0) |   0.5550 |
+------+----------+
| C(1) |   0.4450 |
+------+----------+


In [20]:
marginal_c.values.sum()

np.float64(1.0)

In [21]:
# Query for Joint Distribution P(B, C)
marginal_bc = bp.query(variables=["B", "C"])
print(f"Joint P(B, C): ",marginal_bc)

Joint P(B, C):  +------+------+------------+
| B    | C    |   phi(B,C) |
+======+======+============+
| B(0) | C(0) |     0.3300 |
+------+------+------------+
| B(0) | C(1) |     0.2200 |
+------+------+------------+
| B(1) | C(0) |     0.2250 |
+------+------+------------+
| B(1) | C(1) |     0.2250 |
+------+------+------------+


In [22]:
query_with_evidence = bp.query(
    variables=["D"],
    evidence={"A": 1}
)

print(query_with_evidence)

+------+----------+
| D    |   phi(D) |
+======+==========+
| D(0) |   0.6060 |
+------+----------+
| D(1) |   0.3940 |
+------+----------+


In [25]:
bp.calibrate()
beliefs = bp.get_clique_beliefs()

belief_ab = beliefs[("A", "B")]
belief_bc = beliefs[("B", "C")]

sep_B_from_AB = belief_ab.marginalize(["A"], inplace=False)
sep_B_from_BC = belief_bc.marginalize(["C"], inplace=False)

print("Belief of B from clique (A,B): \n", sep_B_from_AB.values)
print("Belief of B from clique (B,C): \n", sep_B_from_BC.values)

if np.allclose(sep_B_from_AB.values, sep_B_from_BC.values):
    print("Calibration successful: Separator beliefs match.")
else:
    print("Calibration failed.")

Belief of B from clique (A,B): 
 [1.1 0.9]
Belief of B from clique (B,C): 
 [1.1 0.9]
Calibration successful: Separator beliefs match.
